# `parcelsim` — Boston demo

Generates synthetic last-mile parcel delivery demand for **Boston / Suffolk County, MA**
using real US Census ACS 2020 data at census tract level.

Pipeline:
1. Load census tract population & income (US Census ACS 2020)
2. Generate parcel demand — income-stratified USPS model (Yang et al. 2024)
3. Assign parcels to operators — `boston_2024` preset
4. Route with Continuous Approximation (CA) model
5. KPI report

> **First run**: downloads Census data (~10 MB) + TIGER shapefiles (~15 MB).
> Cached to `parcelsim_cache/` after first run.

In [ ]:
!pip install -q "parcelsim[viz,us]"

In [ ]:
import os
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

import parcelsim
os.makedirs('outputs', exist_ok=True)
print(f'parcelsim {parcelsim.__version__}')

# Suffolk County FIPS = 025 (Boston proper)
# Add '017' (Middlesex) or '021' (Norfolk) for Greater Boston
BOROUGH_FIPS = ['025']
STATE        = 'MA'
CRS          = 'EPSG:32619'   # UTM Zone 19N — standard for Boston area

## 1. City

In [ ]:
from parcelsim.city import City

city = City.from_osmnx(
    query='Boston, Massachusetts, USA',
    crs=CRS,
    country_iso='US',
)
print(city)

## 2. Population — US Census ACS 2020

Downloads census tract demographics for Suffolk County.
Add `census_api_key=` if you hit rate limits (free at https://api.census.gov/sign-up.html).

In [ ]:
from parcelsim.population.adapters.census_us import USCensusAdapter

# OBTAIN A FREE API KEY AT https://api.census.gov/sign-up.html AND PUT IT HERE
CENSUS_API_KEY = "YOUR_CENSUS_API_KEY_HERE"

adapter = USCensusAdapter(
    state=STATE,
    county_fips=BOROUGH_FIPS,
    acs_year=2020,
    cache_dir='parcelsim_cache',
    census_api_key=CENSUS_API_KEY,
)

population = adapter.build(city)

print(population.summary())
print(f'\nCensus tracts: {len(city.zones)}')
print(f'Total population:  {city.zones["population"].sum():>10,.0f}')
print(f'Total households:  {city.zones["n_households"].sum():>10,.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
zones_3857 = city.zones.to_crs(epsg=3857)
zones_3857.plot(ax=ax, column='n_households', cmap='YlOrRd', alpha=0.65,
                legend=True, legend_kwds={'label': 'Households / tract', 'shrink': 0.5})
bounds = zones_3857.total_bounds
pad = 2_000
ax.set_xlim(bounds[0]-pad, bounds[2]+pad)
ax.set_ylim(bounds[1]-pad, bounds[3]+pad)
ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.CartoDB.Positron, zoom=12)
ax.set_title('Boston / Suffolk County — Census tracts (ACS 2020)', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('outputs/boston_01_zones.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Parcel demand — income-stratified USPS model

Yang, Landes & Chow (2024) Eq. 1 — weekly parcels per household by income bracket:

| Income bracket | Parcels/hh/week |
|---|---|
| < $35k | 0.50 |
| $35k–$65k | 0.72 |
| $65k–$100k | 0.95 |
| ≥ $100k | 1.25 |

In [ ]:
from parcelsim.demand.usps_model import USPSDemandModel

model = USPSDemandModel(
    volume_increase_factor=1.114,   # 2020→2021 e-commerce growth (Pitney Bowes)
    usps_market_share=0.32,
)

demand = model.generate(population)
print(demand.summary())

In [ ]:
merged = city.zones.merge(demand.zone_demand[['zone_id', 'n_delivery']], on='zone_id')
merged_3857 = merged.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 10))
merged_3857.plot(
    column='n_delivery', ax=ax, cmap='YlOrRd', alpha=0.7,
    legend=True, legend_kwds={'label': 'Parcels/day', 'shrink': 0.5}
)
bounds = merged_3857.total_bounds
pad = 2_000
ax.set_xlim(bounds[0]-pad, bounds[2]+pad)
ax.set_ylim(bounds[1]-pad, bounds[3]+pad)
ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.CartoDB.Positron, zoom=12)
ax.set_title('Boston — Parcel demand by census tract', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('outputs/boston_02_demand.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Operator assignment — boston_2024 preset

In [ ]:
from parcelsim.operators.operator import OperatorRegistry
from parcelsim.operators.assignment import assign_parcels

registry = OperatorRegistry.from_builtin('boston_2024')
assignment = assign_parcels(demand, registry, city)
print(assignment.summary())

## 5. Routing — Continuous Approximation (CA) & KPIs

In [ ]:
from parcelsim.routing.ca.model import CARouter
from parcelsim.output.kpi import KPIReport

router = CARouter()
result = router.solve(assignment, city)
report = KPIReport.from_ca(result, assignment)

w = 30
print(f"{'Metric':<{w}} {'Value':>14}")
print('=' * (w + 16))
print(f"{'Total parcels/day':<{w}} {report.total_parcels_delivered:>14,.0f}")
print(f"{'VKT total (km/day)':<{w}} {report.vkt_total_km:>14,.1f}")
print(f"{'Trucks total':<{w}} {report.n_trucks_total:>14,}")
print(f"{'VKT per parcel (km)':<{w}} {report.vkt_total_km/report.total_parcels_delivered:>14.3f}")
print(report.summary())